# UK Pension Guidance RAG Assistant

A Retrieval-Augmented Generation (RAG) pipeline built on publicly available
UK pension guidance documents.

## What This Does
- Loads and chunks PDF documents from gov.uk, FCA, and The Pensions Regulator
- Embeds chunks using sentence-transformers (all-MiniLM-L6-v2)
- Stores embeddings in ChromaDB (local vector store)
- Answers pension questions grounded in retrieved context using Llama 3.1 via Groq
- Evaluates retrieval quality using a lightweight scoring framework

## Design Decisions
- Model admits uncertainty rather than hallucinating — critical for regulated use cases
- ChromaDB runs locally — no data leaves the environment
- Open-weight model (Llama 3.1) — no vendor lock-in, swappable via one line change
- Evaluation built in from day one — not added after deployment

## Knowledge Base
- FCA Non-Advised Drawdown Pension Sales Review
- Defined Contribution Pension Schemes Explained
- How Your Employer's Pension Scheme Works

## Limitations & Next Steps
- Knowledge base covers 3 documents only — expand for production use
- Evaluation is lightweight — production would use RAGAS framework
- No persistent storage — ChromaDB resets each session
- Would add LangGraph agentic layer for multi-step query handling

In [ ]:
# Simplified stack - avoids LangChain version conflicts entirely
!pip install -q groq chromadb sentence-transformers pypdf langchain-groq langchain-core

In [ ]:
# All imports and setup in one cell to avoid session variable loss
import os
import glob
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader

# ── CONFIG ──────────────────────────────────────────────
GROQ_API_KEY = "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"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ── LOAD PDFs ────────────────────────────────────────────
pdf_files = glob.glob("*.pdf")
if not pdf_files:
    print("No PDFs found - please upload them")
else:
    print(f"Found: {pdf_files}")

    # Extract text from each PDF
    all_chunks = []
    for pdf_path in pdf_files:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and len(text.strip()) > 50:
                # Split page into chunks of ~800 chars
                words = text.split()
                chunk_size = 120  # ~800 chars worth of words
                for i in range(0, len(words), chunk_size - 15):
                    chunk = " ".join(words[i:i + chunk_size])
                    if len(chunk) > 100:
                        all_chunks.append({
                            "text": chunk,
                            "source": pdf_path,
                            "page": page_num + 1
                        })

    print(f"Created {len(all_chunks)} chunks from {len(pdf_files)} PDFs")

    # ── EMBEDDINGS + VECTOR STORE ────────────────────────
    print("Loading embedding model...")
    embedder = SentenceTransformer(EMBEDDING_MODEL)

    # Create ChromaDB collection
    chroma_client = chromadb.Client()

    # Delete collection if exists (handles session restarts)
    try:
        chroma_client.delete_collection("pension_docs")
    except:
        pass

    collection = chroma_client.create_collection("pension_docs")

    # Add chunks to collection in batches
    batch_size = 50
    for i in range(0, len(all_chunks), batch_size):
        batch = all_chunks[i:i + batch_size]
        embeddings = embedder.encode([c["text"] for c in batch]).tolist()
        collection.add(
            documents=[c["text"] for c in batch],
            embeddings=embeddings,
            metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
            ids=[f"chunk_{i+j}" for j, _ in enumerate(batch)]
        )

    print(f"Stored {collection.count()} chunks in ChromaDB")

    # ── GROQ LLM CONNECTION ──────────────────────────────
    client = Groq(api_key=GROQ_API_KEY)

    # Test connection
    test = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "Say OK"}],
        max_tokens=5
    )
    print(f"LLM connected: {test.choices[0].message.content}")

In [ ]:
# ── RAG QUERY FUNCTION ───────────────────────────────────────────────────────
def ask_pension_assistant(question, top_k=4):
    """
    Full RAG pipeline:
    1. Embed the question
    2. Retrieve top_k relevant chunks from ChromaDB
    3. Send context + question to LLM
    4. Return grounded answer with sources

    Design decision: k=4 balances context richness against prompt length.
    Too few chunks = missing relevant info. Too many = context dilution.
    """
    # Step 1: embed the question
    query_embedding = embedder.encode([question]).tolist()

    # Step 2: retrieve relevant chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    context = "\n\n".join(results["documents"][0])
    sources = [f"{m['source']} p.{m['page']}"
               for m in results["metadatas"][0]]

    # Step 3: build prompt - explicitly grounded, admits uncertainty
    # Critical design decision for regulated use case:
    # model must not hallucinate when context is insufficient
    prompt = f"""You are a UK pension guidance assistant.
Answer the question using ONLY the context below.
If the answer is not clearly in the context, say:
"I don't have enough information in my knowledge base to answer this accurately."

Context:
{context}

Question: {question}

Answer:"""

    # Step 4: call LLM
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,  # low = more factual, less creative
        max_tokens=400
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": sources
    }


# ── EVALUATION FUNCTION ──────────────────────────────────────────────────────
def evaluate_retrieval(question, context):
    """
    Lightweight retrieval relevance scoring.
    Simplified version of the RAGAS context_relevance metric.

    Design decision: systematic evaluation must be built into the pipeline
    from day one - not added after deployment. In a regulated environment
    like L&G, you need documented evidence that the system was tested
    rigorously before go-live.
    """
    eval_prompt = f"""Rate how relevant this context is to the question.
Score 1-5: 1=irrelevant, 3=partial, 5=directly answers it.
Respond ONLY as: SCORE: [1-5] | REASON: [one sentence]

Question: {question}
Context (first 300 chars): {context[:300]}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0,
        max_tokens=60
    )
    return response.choices[0].message.content


# ── TEST SUITE ───────────────────────────────────────────────────────────────
test_questions = [
    "What is pension drawdown and what are the risks?",
    "How much does my employer have to contribute to my pension?",
    "What happens to my pension if I die before retiring?",
    "Can I take my pension as a lump sum?",
    "What is the difference between defined benefit and defined contribution?"
]

print("=" * 65)
print("UK PENSION GUIDANCE RAG ASSISTANT — TEST SUITE WITH EVALUATION")
print("=" * 65)

for i, question in enumerate(test_questions, 1):
    result = ask_pension_assistant(question)

    # Get context for evaluation
    query_embedding = embedder.encode([question]).tolist()
    retrieved = collection.query(query_embeddings=query_embedding, n_results=1)
    context_sample = retrieved["documents"][0][0] if retrieved["documents"][0] else ""
    eval_score = evaluate_retrieval(question, context_sample)

    print(f"\nQ{i}: {question}")
    print(f"{'─' * 50}")
    print(f"Answer: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print(f"Eval:    {eval_score}")

print("\n" + "=" * 65)
print("END OF TEST SUITE")
print("=" * 65)